In [ ]:
pip install xgboost


In [ ]:
pip install lightgbm

In [ ]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.8 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/calories.csv')

In [ ]:
df.head(3)

,User_ID,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,14733363,male,68,190.0,94.0,29.0,105.0,40.8,231.0
1,14861698,female,20,166.0,60.0,14.0,94.0,40.3,66.0
2,11179863,male,69,179.0,79.0,5.0,88.0,38.7,26.0


In [ ]:
df.drop(['User_ID'],axis=1,inplace=True)

In [ ]:
df.head(3)

,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,male,68,190.0,94.0,29.0,105.0,40.8,231.0
1,female,20,166.0,60.0,14.0,94.0,40.3,66.0
2,male,69,179.0,79.0,5.0,88.0,38.7,26.0


In [ ]:
df.isnull().sum()

,0
Gender,0
Age,0
Height,0
Weight,0
Duration,0
Heart_Rate,0
Body_Temp,0
Calories,0


In [ ]:
x=df.drop('Calories',axis=1)
y=df['Calories']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42)

In [ ]:
preprocessor=ColumnTransformer(transformers=[
    ('tnf1',OneHotEncoder(sparse_output=False,drop='first'),['Gender'])
], remainder='passthrough')

In [ ]:
x_train=preprocessor.fit_transform(x_train)


In [ ]:
x_test=preprocessor.transform(x_test)

In [ ]:
pipe_linreg = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
pipe_rfr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor())
])
pipe_xgbr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBRegressor())
])
pipe_lgbmr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LGBMRegressor())
])
pipe_cbr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', CatBoostRegressor())
])


In [ ]:
model_hyperparameters = {

    pipe_linreg: {},

    pipe_rfr: {
        'model__n_estimators': [50,100],
        'model__max_depth': [None, 10, 20]

    },

    pipe_xgbr: {
        'model__n_estimators': [50,100],
        'model__learning_rate': [0.01, 0.1]
    },

    pipe_lgbmr: {
        'model__n_estimators': [50,100],
        'model__learning_rate': [0.01, 0.1]

    },

    pipe_cbr: {
        'model__iterations': [100],
        'model__learning_rate': [0.05, 0.1]
    }
}

In [ ]:
def ModelSelection(model_hyperparameters, X_train, y_train):
    results = []
    best_model = None
    best_score = -float('inf') # স্কোর নেগেটিভও হতে পারে তাই inf রাখা নিরাপদ

    for model, params in model_hyperparameters.items():
        # scoring='r2' ব্যবহার করা হয়েছে
        grid = GridSearchCV(model, params, cv=3, scoring='r2')
        grid.fit(X_train, y_train)
        score = grid.best_score_

        results.append({
            "model": model,
            "best_score": score,
            "best_params": grid.best_params_
        })

        if score > best_score:
            best_score = score
            best_model = grid.best_estimator_

    results_df = pd.DataFrame(results)
    return results_df, best_model

In [ ]:
results_df, best_model = ModelSelection(model_hyperparameters, x_train, y_train)
print(results_df)

print("Best Model:", best_model)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000627 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 358
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 7
[LightGBM] [Info] Start training from score 89.078750
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000207 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 358
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 7
[LightGBM] [Info] Start training from score 89.078125
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000631 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 358
[LightGBM] [Info] Number of data points in the train set: 800

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000582 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 358
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 7
[LightGBM] [Info] Start training from score 89.078750
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000585 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 358
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 7
[LightGBM] [Info] Start training from score 89.078125


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000639 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 358
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 7
[LightGBM] [Info] Start training from score 89.129375
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001053 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 358
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 7
[LightGBM] [Info] Start training from score 89.078750


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000590 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 358
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 7
[LightGBM] [Info] Start training from score 89.078125
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000601 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 358
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 7
[LightGBM] [Info] Start training from score 89.129375
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000194 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 358
[LightGBM] [Info] Number of data points in the train set: 800

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000594 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 358
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 7
[LightGBM] [Info] Start training from score 89.078125
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000861 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 358
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 7
[LightGBM] [Info] Start training from score 89.129375


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000877 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 366
[LightGBM] [Info] Number of data points in the train set: 12000, number of used features: 7
[LightGBM] [Info] Start training from score 89.095417
0:	learn: 59.9358037	total: 2.35ms	remaining: 233ms
1:	learn: 57.2404989	total: 4.27ms	remaining: 209ms
2:	learn: 54.7183223	total: 6.14ms	remaining: 199ms
3:	learn: 52.3166374	total: 7.88ms	remaining: 189ms
4:	learn: 49.9793335	total: 9.7ms	remaining: 184ms
5:	learn: 47.8328739	total: 11.4ms	remaining: 179ms
6:	learn: 45.8165925	total: 12.9ms	remaining: 171ms
7:	learn: 43.9168955	total: 14.6ms	remaining: 168ms
8:	learn: 42.0132931	total: 16.4ms	remaining: 166ms
9:	learn: 40.1955787	total: 18.2ms	remaining: 164ms
10:	learn: 38.4767051	total: 19.9ms	remaining: 161ms
11:	learn: 36.8447663	total: 21.6ms	remaining: 158ms
12:	learn: 35.2613550	total: 23.4m

In [ ]:
print("Best Model:", best_model)

Best Model: Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=None, device=None,
                              early_stopping_rounds=None,
                              enable_categorical=False, eval_metric=None,
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.1,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                             

In [ ]:
y_pred = best_model.predict(x_test)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print("R-squared (R2) score:", r2_score(y_test, y_pred))
print("Mean Absolute Error (MAE):", mean_absolute_error(y_test, y_pred))
print("Mean Squared Error (MSE):", mean_squared_error(y_test, y_pred))

R-squared (R2) score: 0.9992399914008103
Mean Absolute Error (MAE): 1.2241583960056306
Mean Squared Error (MSE): 3.0672362273095497


In [ ]:
from sklearn.metrics import r2_score
y_pred = best_model.predict(x_test)
print("Test Set R2 Score:", r2_score(y_test, y_pred))

Test Set R2 Score: 0.9992399914008103


In [ ]:
import pandas as pd
import numpy as np

person_data = {
    'User_ID': [123456],
    'Gender': ['male'],
    'Age': [10],
    'Height': [152.4],
    'Weight': [45.0],
    'Duration': [20.0],
    'Heart_Rate': [110.0],
    'Body_Temp': [40.2]
}


sample_df = pd.DataFrame(person_data)


sample_processed = preprocessor.transform(sample_df.drop(columns=['User_ID']))


predicted_calories = best_model.predict(sample_processed)

print("="*30)
print("   CALORIES PREDICTION RESULT   ")
print("="*30)
print(f"Gender     : {person_data['Gender'][0]}")
print(f"Age        : {person_data['Age'][0]} years")
print(f"Duration   : {person_data['Duration'][0]} minutes")
print(f"Heart Rate : {person_data['Heart_Rate'][0]} bpm")
print("-" * 30)
print(f"Predicted Calories Burned: {predicted_calories[0]:.2f} kcal")
print("="*30)

   CALORIES PREDICTION RESULT   
Gender     : male
Age        : 10 years
Duration   : 20.0 minutes
Heart Rate : 110.0 bpm
------------------------------
Predicted Calories Burned: 120.85 kcal
